# Notebook 03 — Model Training MLflow

## Imports

In [1]:
import os
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow
import mlflow.sklearn

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    RocCurveDisplay,
    ConfusionMatrixDisplay,
    classification_report,
)
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    RandomizedSearchCV,
    GridSearchCV,
    cross_val_score
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
TARGET = "TARGET"
CLIENT_ID = "SK_ID_CURR"

pd.set_option("display.max_columns", 200)

/Users/marinramananjaona/Projet Openclassrooms/Projet 6/mlops-credit-scoring/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Configurer MLflow

In [2]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")

experiment_name = "credit_scoring_training"

mlflow.set_experiment(experiment_name)

print("Tracking URI :", mlflow.get_tracking_uri())
print("Expérience   :", experiment_name)

Tracking URI : http://127.0.0.1:5000
Expérience   : credit_scoring_training


## Charger Dataset

In [3]:
train = pd.read_parquet(
    "../data/processed/application_train_enriched.parquet"
)

test = pd.read_parquet(
    "../data/processed/application_test_enriched.parquet"
)

print("Train :", train.shape)
print("Test  :", test.shape)

Train : (307511, 543)
Test  : (48744, 542)


## Préparer X et y

In [4]:
X = train.drop(
    columns=[TARGET, CLIENT_ID]
)

y = train[TARGET].astype("int8")

In [5]:
# prédictions Kaggle
X_test = test.drop(
    columns=[CLIENT_ID]
)

test_ids = test[CLIENT_ID].copy()

In [6]:
print("X :", X.shape)
print("y :", y.shape)

print(y.value_counts())
print(y.value_counts(normalize=True))

X : (307511, 541)
y : (307511,)
TARGET
0    282686
1     24825
Name: count, dtype: int64
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64


## Convertir booléens en nombres
Après get_dummies, certaines colonnes peuvent être de type booléen.

In [7]:
bool_columns = X.select_dtypes(
    include="bool"
).columns

X[bool_columns] = X[bool_columns].astype("int8")
X_test[bool_columns] = X_test[bool_columns].astype("int8")

print("Colonnes booléennes converties :", len(bool_columns))

Colonnes booléennes converties : 131


In [8]:
print(X.dtypes.value_counts())

float64    367
int8       131
int64       43
Name: count, dtype: int64


## Split
Utilise une séparation stratifiée pour préserver la proportion de clients en défaut.

In [9]:
X_train, X_valid, y_train, y_valid = train_test_split(

    X,
    y,

    test_size=0.2,

    stratify=y,

    random_state=RANDOM_STATE
)

In [10]:
print("Train :")
print(y_train.value_counts(normalize=True))

print("\nValidation :")
print(y_valid.value_counts(normalize=True))

Train :
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64

Validation :
TARGET
0    0.919272
1    0.080728
Name: proportion, dtype: float64


## Définir le score métier
- FP = bon client prédit mauvais
- FN = mauvais client prédit bon
- coût = FP + 10 × FN

In [11]:
def compute_business_cost(
    y_true,
    y_pred,
    false_positive_cost=1,
    false_negative_cost=10
):
    """
    Calcule le coût métier total.

    FP : bon client prédit en défaut.
    FN : client en défaut prédit comme bon client.
    """

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    total_cost = (
        false_positive_cost * fp
        + false_negative_cost * fn
    )

    return {
        "business_cost": int(total_cost),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    }

In [12]:
# Coût normalisé
def normalized_business_cost(
    y_true,
    y_pred,
    false_positive_cost=1,
    false_negative_cost=10
):
    result = compute_business_cost(
        y_true,
        y_pred,
        false_positive_cost,
        false_negative_cost
    )

    return result["business_cost"] / len(y_true)

## Définir une fonction d’évaluation

In [13]:
def evaluate_classifier(
    model,
    X_eval,
    y_eval,
    threshold=0.5
):
    """
    Évalue un modèle à partir des probabilités prédites.
    """

    probabilities = model.predict_proba(
        X_eval
    )[:, 1]

    predictions = (
        probabilities >= threshold
    ).astype("int8")

    business = compute_business_cost(
        y_eval,
        predictions
    )

    metrics = {
        "threshold": float(threshold),
        "roc_auc": float(
            roc_auc_score(
                y_eval,
                probabilities
            )
        ),
        "accuracy": float(
            accuracy_score(
                y_eval,
                predictions
            )
        ),
        "precision": float(
            precision_score(
                y_eval,
                predictions,
                zero_division=0
            )
        ),
        "recall": float(
            recall_score(
                y_eval,
                predictions,
                zero_division=0
            )
        ),
        "f1": float(
            f1_score(
                y_eval,
                predictions,
                zero_division=0
            )
        ),
        "business_cost": business["business_cost"],
        "normalized_business_cost": (
            business["business_cost"]
            / len(y_eval)
        ),
        "tn": business["tn"],
        "fp": business["fp"],
        "fn": business["fn"],
        "tp": business["tp"]
    }

    return metrics, probabilities, predictions

## Créer une fonction d’optimisation du seuil
Elle recherche le seuil qui minimise le coût métier.

In [14]:
def find_best_threshold(
    y_true,
    probabilities,
    thresholds=None,
    false_positive_cost=1,
    false_negative_cost=10
):
    """
    Teste plusieurs seuils et retourne celui qui minimise
    le coût métier.
    """

    if thresholds is None:
        thresholds = np.arange(
            0.01,
            0.81,
            0.01
        )

    results = []

    for threshold in thresholds:
        predictions = (
            probabilities >= threshold
        ).astype("int8")

        business = compute_business_cost(
            y_true,
            predictions,
            false_positive_cost,
            false_negative_cost
        )

        results.append({
            "threshold": float(threshold),
            "business_cost": business["business_cost"],
            "normalized_business_cost": (
                business["business_cost"]
                / len(y_true)
            ),
            "fp": business["fp"],
            "fn": business["fn"],
            "tp": business["tp"],
            "tn": business["tn"]
        })

    results_df = pd.DataFrame(results)

    best_row = results_df.loc[
        results_df["business_cost"].idxmin()
    ]

    return best_row, results_df

## Fonction de création des graphiques

In [15]:
def save_evaluation_plots(
    y_true,
    probabilities,
    predictions,
    threshold_results,
    output_dir,
    prefix
):
    """
    Sauvegarde les graphiques qui seront ajoutés comme artefacts MLflow.
    """

    output_dir = Path(output_dir)
    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # Courbe ROC
    fig, ax = plt.subplots(figsize=(7, 5))

    RocCurveDisplay.from_predictions(
        y_true,
        probabilities,
        ax=ax
    )

    ax.set_title(f"Courbe ROC — {prefix}")

    roc_path = output_dir / f"{prefix}_roc_curve.png"

    fig.tight_layout()
    fig.savefig(
        roc_path,
        dpi=150
    )

    plt.close(fig)

    # Matrice de confusion
    fig, ax = plt.subplots(figsize=(6, 5))

    ConfusionMatrixDisplay.from_predictions(
        y_true,
        predictions,
        ax=ax,
        values_format="d"
    )

    ax.set_title(
        f"Matrice de confusion — {prefix}"
    )

    confusion_path = (
        output_dir
        / f"{prefix}_confusion_matrix.png"
    )

    fig.tight_layout()
    fig.savefig(
        confusion_path,
        dpi=150
    )

    plt.close(fig)

    # Coût métier selon le seuil
    fig, ax = plt.subplots(figsize=(8, 5))

    ax.plot(
        threshold_results["threshold"],
        threshold_results["business_cost"]
    )

    ax.set_xlabel("Seuil")
    ax.set_ylabel("Coût métier")
    ax.set_title(
        f"Coût métier selon le seuil — {prefix}"
    )

    threshold_path = (
        output_dir
        / f"{prefix}_threshold_cost.png"
    )

    fig.tight_layout()
    fig.savefig(
        threshold_path,
        dpi=150
    )

    plt.close(fig)

    return {
        "roc_curve": roc_path,
        "confusion_matrix": confusion_path,
        "threshold_cost": threshold_path
    }

## Préparer le dossier temporaire d’artefacts

In [16]:
ARTIFACTS_DIR = Path("../reports/mlflow_artifacts")

ARTIFACTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

## Préprocessing

### Run 1 — Baseline naïve
Une baseline naïve permet de vérifier qu’un vrai modèle fait mieux qu’une stratégie triviale.

In [17]:
dummy_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "model",
            DummyClassifier(
                strategy="prior",
                random_state=RANDOM_STATE
            )
        )
    ]
)

In [18]:
# Entraînement et tracking :

with mlflow.start_run(
    run_name="01_dummy_baseline"
) as run:

    mlflow.set_tags({
        "project": "credit_scoring",
        "model_family": "baseline",
        "dataset_version": "train_features_v1",
        "author": "Marin",
        "description": (
            "Baseline naïve prédisant selon "
            "la distribution des classes."
        )
    })

    start_time = time.time()

    dummy_pipeline.fit(
        X_train,
        y_train
    )

    training_time = time.time() - start_time

    metrics_05, probabilities, predictions = (
        evaluate_classifier(
            dummy_pipeline,
            X_valid,
            y_valid,
            threshold=0.5
        )
    )

    best_threshold, threshold_results = (
        find_best_threshold(
            y_valid,
            probabilities
        )
    )

    mlflow.log_params({
        "model": "DummyClassifier",
        "strategy": "prior",
        "random_state": RANDOM_STATE,
        "train_rows": len(X_train),
        "valid_rows": len(X_valid),
        "n_features": X_train.shape[1]
    })

    mlflow.log_metrics({
        **metrics_05,
        "training_time_seconds": training_time,
        "best_threshold": float(
            best_threshold["threshold"]
        ),
        "best_business_cost": float(
            best_threshold["business_cost"]
        ),
        "best_normalized_business_cost": float(
            best_threshold[
                "normalized_business_cost"
            ]
        )
    })

    threshold_results.to_csv(
        ARTIFACTS_DIR
        / "dummy_threshold_results.csv",
        index=False
    )

    mlflow.log_artifact(
        str(
            ARTIFACTS_DIR
            / "dummy_threshold_results.csv"
        ),
        artifact_path="evaluation"
    )

    mlflow.sklearn.log_model(
        sk_model=dummy_pipeline,
        name="model",
        input_example=X_valid.head(5)
    )

    print("Run ID :", run.info.run_id)

Run ID : c67a736aa0074b899cba540d54c9210a
🏃 View run 01_dummy_baseline at: http://127.0.0.1:5000/#/experiments/1/runs/c67a736aa0074b899cba540d54c9210a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


### Run 2 — Régression logistique
La régression logistique constitue une baseline interprétable.

In [19]:
logistic_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True
            )
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(
                C=1.0,
                class_weight="balanced",
                max_iter=1000,
                solver="liblinear",
                random_state=RANDOM_STATE
            )
        )
    ]
)

Le pipeline garantit que l’imputation et la standardisation sont apprises uniquement sur les données utilisées pour l’entraînement.

In [20]:
with mlflow.start_run(
    run_name="02_logistic_regression_baseline"
) as run:

    mlflow.set_tags({
        "project": "credit_scoring",
        "model_family": "linear_model",
        "dataset_version": "train_features_v1",
        "imbalance_strategy": "class_weight_balanced",
        "description": (
            "Régression logistique avec imputation médiane, "
            "indicateurs de valeurs manquantes et standardisation."
        )
    })

    params = {
        "model": "LogisticRegression",
        "C": 1.0,
        "class_weight": "balanced",
        "max_iter": 1000,
        "solver": "liblinear",
        "imputation": "median",
        "missing_indicator": True,
        "scaling": "StandardScaler",
        "random_state": RANDOM_STATE,
        "n_features": X_train.shape[1]
    }

    mlflow.log_params(params)

    start_time = time.time()

    logistic_pipeline.fit(
        X_train,
        y_train
    )

    training_time = time.time() - start_time

    metrics_05, probabilities, predictions_05 = (
        evaluate_classifier(
            logistic_pipeline,
            X_valid,
            y_valid,
            threshold=0.5
        )
    )

    best_threshold, threshold_results = (
        find_best_threshold(
            y_valid,
            probabilities
        )
    )

    optimal_threshold = float(
        best_threshold["threshold"]
    )

    metrics_optimal, _, predictions_optimal = (
        evaluate_classifier(
            logistic_pipeline,
            X_valid,
            y_valid,
            threshold=optimal_threshold
        )
    )

    mlflow.log_metrics({
        **{
            f"default_{key}": value
            for key, value in metrics_05.items()
        },
        **{
            f"optimal_{key}": value
            for key, value in metrics_optimal.items()
        },
        "training_time_seconds": training_time
    })

    threshold_csv = (
        ARTIFACTS_DIR
        / "logistic_threshold_results.csv"
    )

    threshold_results.to_csv(
        threshold_csv,
        index=False
    )

    plot_paths = save_evaluation_plots(
        y_true=y_valid,
        probabilities=probabilities,
        predictions=predictions_optimal,
        threshold_results=threshold_results,
        output_dir=ARTIFACTS_DIR,
        prefix="logistic"
    )

    mlflow.log_artifact(
        str(threshold_csv),
        artifact_path="evaluation"
    )

    for plot_path in plot_paths.values():
        mlflow.log_artifact(
            str(plot_path),
            artifact_path="plots"
        )

    mlflow.sklearn.log_model(
        sk_model=logistic_pipeline,
        name="model",
        input_example=X_valid.head(5)
    )

    print("Run ID :", run.info.run_id)
    print("AUC :", metrics_05["roc_auc"])
    print("Seuil optimal :", optimal_threshold)
    print(
        "Coût métier optimal :",
        metrics_optimal["business_cost"]
    )

Run ID : be44797797b84be2ae9367a5eebfcc42
AUC : 0.77560833792257
Seuil optimal : 0.54
Coût métier optimal : 30944
🏃 View run 02_logistic_regression_baseline at: http://127.0.0.1:5000/#/experiments/1/runs/be44797797b84be2ae9367a5eebfcc42
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


### Run 3 — Validation croisée de la régression logistique
La validation croisée donne une estimation plus stable que celle obtenue sur un seul split.

In [21]:
#cv = StratifiedKFold(
#    n_splits=5,
#    shuffle=True,
#    random_state=RANDOM_STATE
#)

In [22]:
# Pour limiter le temps de calcul, tu peux commencer avec 3 plis :
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

In [23]:
with mlflow.start_run(
    run_name="03_logistic_regression_cross_validation"
):

    mlflow.set_tags({
        "project": "credit_scoring",
        "model_family": "linear_model",
        "validation_strategy": "StratifiedKFold",
        "description": (
            "Validation croisée stratifiée de la "
            "régression logistique."
        )
    })

    start_time = time.time()

    cv_results = cross_validate(
        estimator=logistic_pipeline,
        X=X_train,
        y=y_train,
        cv=cv,
        scoring={
            "roc_auc": "roc_auc",
            "accuracy": "accuracy",
            "recall": "recall",
            "precision": "precision"
        },
        n_jobs=-1,
        return_train_score=False
    )

    cv_time = time.time() - start_time

    metrics_to_log = {
        "cv_roc_auc_mean": (
            cv_results["test_roc_auc"].mean()
        ),
        "cv_roc_auc_std": (
            cv_results["test_roc_auc"].std()
        ),
        "cv_accuracy_mean": (
            cv_results["test_accuracy"].mean()
        ),
        "cv_recall_mean": (
            cv_results["test_recall"].mean()
        ),
        "cv_precision_mean": (
            cv_results["test_precision"].mean()
        ),
        "cv_total_time_seconds": cv_time
    }

    mlflow.log_params({
        "model": "LogisticRegression",
        "cv_folds": cv.get_n_splits(),
        "cv_type": "StratifiedKFold",
        "shuffle": True,
        "random_state": RANDOM_STATE
    })

    mlflow.log_metrics(metrics_to_log)

    display(
        pd.DataFrame(cv_results)
    )

/Users/marinramananjaona/Projet Openclassrooms/Projet 6/mlops-credit-scoring/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/marinramananjaona/Projet Openclassrooms/Projet 6/mlops-credit-scoring/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/marinramananjaona/Projet Openclassrooms/Projet 6/mlops-credit-scoring/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/marinramananjaona/Projet Openclassrooms/Projet 6/mlops-credit-scoring/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/marinramananjaona/Projet Openclassrooms/Projet 6/mlops-credit-scoring/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encounter

,fit_time,score_time,test_roc_auc,test_accuracy,test_recall,test_precision
0,61556.298938,7.331784,0.770642,0.708718,0.700000,0.174644
1,61672.434597,2.071489,0.775023,0.710681,0.701511,0.175956
2,62220.841205,5.149280,0.771464,0.708300,0.695317,0.173659


🏃 View run 03_logistic_regression_cross_validation at: http://127.0.0.1:5000/#/experiments/1/runs/6f743c19e4564e24acaa2935e14ab1b7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


cross_validate permet de calculer plusieurs métriques sur chaque pli, tandis que StratifiedKFold préserve la répartition des classes.

### Run 4 — LightGBM baseline
LightGBM peut gérer nativement les valeurs manquantes. Il n’a donc pas besoin du même pipeline que la régression logistique.

In [26]:
lgbm_model = LGBMClassifier(
    objective="binary",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1
)

In [31]:
import re

def clean_column_names(df):
    df = df.copy()

    df.columns = [
        re.sub(r'[^A-Za-z0-9_]', '_', col)
        for col in df.columns
    ]

    return df

X_train = clean_column_names(X_train)
X_valid = clean_column_names(X_valid)
X_test = clean_column_names(X_test)

In [32]:
with mlflow.start_run(
    run_name="04_lightgbm_baseline"
) as run:

    mlflow.set_tags({
        "project": "credit_scoring",
        "model_family": "gradient_boosting",
        "dataset_version": "train_features_v1",
        "imbalance_strategy": "class_weight_balanced",
        "description": (
            "LightGBM baseline avec gestion native "
            "des valeurs manquantes."
        )
    })

    params = lgbm_model.get_params()

    # MLflow accepte mieux des paramètres simples
    params_to_log = {
        key: str(value)
        if value is None
        else value
        for key, value in params.items()
    }

    mlflow.log_params(params_to_log)

    start_time = time.time()

    lgbm_model.fit(
        X_train,
        y_train
    )

    training_time = time.time() - start_time

    metrics_05, probabilities, predictions_05 = (
        evaluate_classifier(
            lgbm_model,
            X_valid,
            y_valid,
            threshold=0.5
        )
    )

    best_threshold, threshold_results = (
        find_best_threshold(
            y_valid,
            probabilities
        )
    )

    optimal_threshold = float(
        best_threshold["threshold"]
    )

    metrics_optimal, _, predictions_optimal = (
        evaluate_classifier(
            lgbm_model,
            X_valid,
            y_valid,
            threshold=optimal_threshold
        )
    )

    mlflow.log_metrics({
        **{
            f"default_{key}": value
            for key, value in metrics_05.items()
        },
        **{
            f"optimal_{key}": value
            for key, value
            in metrics_optimal.items()
        },
        "training_time_seconds": training_time
    })

    threshold_csv = (
        ARTIFACTS_DIR
        / "lightgbm_threshold_results.csv"
    )

    threshold_results.to_csv(
        threshold_csv,
        index=False
    )

    plot_paths = save_evaluation_plots(
        y_true=y_valid,
        probabilities=probabilities,
        predictions=predictions_optimal,
        threshold_results=threshold_results,
        output_dir=ARTIFACTS_DIR,
        prefix="lightgbm"
    )

    mlflow.log_artifact(
        str(threshold_csv),
        artifact_path="evaluation"
    )

    for plot_path in plot_paths.values():
        mlflow.log_artifact(
            str(plot_path),
            artifact_path="plots"
        )

    mlflow.sklearn.log_model(
        sk_model=lgbm_model,
        name="model",
        input_example=X_valid.head(5)
    )

    print("Run ID :", run.info.run_id)
    print("AUC :", metrics_05["roc_auc"])
    print("Seuil optimal :", optimal_threshold)
    print(
        "Coût métier optimal :",
        metrics_optimal["business_cost"]
    )

Run ID : 928dd53fc9024826b949993cabc163ee
AUC : 0.790132832975617
Seuil optimal : 0.49
Coût métier optimal : 29585
🏃 View run 04_lightgbm_baseline at: http://127.0.0.1:5000/#/experiments/1/runs/928dd53fc9024826b949993cabc163ee
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


### Run 5 — Comparer la gestion du déséquilibre
Teste au moins deux variantes :
- class_weight=None
- class_weight="balanced"

In [33]:
imbalance_strategies = [
    None,
    "balanced"
]

for strategy in imbalance_strategies:

    model = LGBMClassifier(
        objective="binary",
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        class_weight=strategy,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1
    )

    run_name = (
        f"05_lightgbm_class_weight_{strategy}"
    )

    with mlflow.start_run(
        run_name=run_name
    ):

        mlflow.set_tags({
            "model_family": "gradient_boosting",
            "experiment_type": (
                "imbalance_strategy_comparison"
            )
        })

        model.fit(
            X_train,
            y_train
        )

        probabilities = model.predict_proba(
            X_valid
        )[:, 1]

        best_threshold, threshold_results = (
            find_best_threshold(
                y_valid,
                probabilities
            )
        )

        optimal_threshold = float(
            best_threshold["threshold"]
        )

        metrics, _, _ = evaluate_classifier(
            model,
            X_valid,
            y_valid,
            threshold=optimal_threshold
        )

        mlflow.log_param(
            "class_weight",
            str(strategy)
        )

        mlflow.log_metrics({
            f"optimal_{key}": value
            for key, value in metrics.items()
        })

🏃 View run 05_lightgbm_class_weight_None at: http://127.0.0.1:5000/#/experiments/1/runs/8aad1493d4874d75a48f3b6fb5a659ab
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run 05_lightgbm_class_weight_balanced at: http://127.0.0.1:5000/#/experiments/1/runs/660b52f2da11487cb635f951cb612f33
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


### Optimisation des hyperparamètres
Sur 307 000 lignes, une grosse GridSearchCV peut être très longue. Commence avec RandomizedSearchCV, qui teste un nombre limité de combinaisons.

In [34]:
from scipy.stats import randint, uniform

lgbm_search_model = LGBMClassifier(
    objective="binary",
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1
)

param_distributions = {
    "n_estimators": randint(200, 900),
    "learning_rate": uniform(0.02, 0.12),
    "num_leaves": randint(15, 80),
    "max_depth": [-1, 5, 8, 12],
    "min_child_samples": randint(20, 150),
    "subsample": uniform(0.65, 0.35),
    "colsample_bytree": uniform(0.65, 0.35),
    "reg_alpha": uniform(0.0, 2.0),
    "reg_lambda": uniform(0.0, 2.0)
}

In [35]:
search_cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

random_search = RandomizedSearchCV(
    estimator=lgbm_search_model,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="roc_auc",
    cv=search_cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=2,
    return_train_score=True
)

RandomizedSearchCV explore un nombre fixé de combinaisons, tandis que GridSearchCV teste exhaustivement toutes les combinaisons spécifiées.

### Run 6 — Tracker la recherche d’hyperparamètres

In [36]:
with mlflow.start_run(
    run_name="06_lightgbm_randomized_search"
) as parent_run:

    mlflow.set_tags({
        "project": "credit_scoring",
        "model_family": "gradient_boosting",
        "experiment_type": "hyperparameter_search",
        "search_method": "RandomizedSearchCV",
        "description": (
            "Recherche de 20 configurations LightGBM "
            "avec validation croisée stratifiée."
        )
    })

    mlflow.log_params({
        "search_method": "RandomizedSearchCV",
        "n_iter": 20,
        "cv_folds": 3,
        "scoring": "roc_auc",
        "random_state": RANDOM_STATE
    })

    start_time = time.time()

    random_search.fit(
        X_train,
        y_train
    )

    search_time = time.time() - start_time

    mlflow.log_metric(
        "search_time_seconds",
        search_time
    )

    mlflow.log_metric(
        "best_cv_roc_auc",
        random_search.best_score_
    )

    mlflow.log_params({
        f"best_{key}": value
        for key, value
        in random_search.best_params_.items()
    })

    search_results = pd.DataFrame(
        random_search.cv_results_
    )

    search_results_path = (
        ARTIFACTS_DIR
        / "lightgbm_random_search_results.csv"
    )

    search_results.to_csv(
        search_results_path,
        index=False
    )

    mlflow.log_artifact(
        str(search_results_path),
        artifact_path="hyperparameter_search"
    )

    best_search_model = (
        random_search.best_estimator_
    )

    probabilities = (
        best_search_model.predict_proba(
            X_valid
        )[:, 1]
    )

    best_threshold, threshold_results = (
        find_best_threshold(
            y_valid,
            probabilities
        )
    )

    optimal_threshold = float(
        best_threshold["threshold"]
    )

    metrics, _, predictions = (
        evaluate_classifier(
            best_search_model,
            X_valid,
            y_valid,
            threshold=optimal_threshold
        )
    )

    mlflow.log_metrics({
        f"validation_{key}": value
        for key, value in metrics.items()
    })

    mlflow.sklearn.log_model(
        sk_model=best_search_model,
        name="model",
        input_example=X_valid.head(5)
    )

    print("Meilleurs paramètres :")
    print(random_search.best_params_)

    print("\nMeilleure AUC CV :")
    print(random_search.best_score_)

    print("\nSeuil optimal :")
    print(optimal_threshold)

Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END colsample_bytree=0.9784934481555125, learning_rate=0.02009345190092172, max_depth=12, min_child_samples=40, n_estimators=360, num_leaves=72, reg_alpha=1.0495128632644757, reg_lambda=0.8638900372842315, subsample=0.7519301990693147; total time= 3.1min
[CV] END colsample_bytree=0.9212964881763901, learning_rate=0.13273987298770268, max_depth=5, min_child_samples=33, n_estimators=470, num_leaves=76, reg_alpha=0.6506606615265287, reg_lambda=0.777354579378964, subsample=0.7449721611208636; total time= 1.6min
[CV] END colsample_bytree=0.8641485131528328, learning_rate=0.03673926327824502, max_depth=12, min_child_samples=34, n_estimators=389, num_leaves=76, reg_alpha=1.5703519227860272, reg_lambda=0.39934756431671947, subsample=0.8299820534447641; total time= 3.0min
[CV] END colsample_bytree=0.9212964881763901, learning_rate=0.13273987298770268, max_depth=5, min_child_samples=33, n_estimators=470, num_leaves=76, reg_alpha=0

Les runs parent-enfants sont également une possibilité utile lorsque chaque configuration doit apparaître comme un run distinct dans MLflow.

### Attention à l’optimisation du seuil
Pour un projet rigoureux, évite d’utiliser le même jeu pour :

1. sélectionner les hyperparamètres ;
2. sélectionner le seuil ;
3. annoncer les performances finales.

Une meilleure organisation est :

In [ ]:
# train
# ├── sous-train : entraînement
# ├── validation : hyperparamètres et seuil
# └── test interne : évaluation finale

In [37]:
# Créer 3 ensembles :
X_temp, X_holdout, y_temp, y_holdout = train_test_split(
    X,
    y,
    test_size=0.15,
    stratify=y,
    random_state=RANDOM_STATE
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp,
    y_temp,
    test_size=0.1765,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

In [45]:
X_train = clean_column_names(X_train)
X_valid = clean_column_names(X_valid)

Cela donne approximativement :

- 70 % entraînement
- 15 % validation
- 15 % holdout final

Tu utilises :

- X_train pour l’entraînement ;
- X_valid pour le seuil ;
- X_holdout une seule fois pour la performance finale.

### Évaluation finale sur le holdout

Après avoir sélectionné le meilleur modèle et le meilleur seuil :

In [46]:
final_model = random_search.best_estimator_

final_model.fit(
    pd.concat([X_train, X_valid]),
    pd.concat([y_train, y_valid])
)

LGBMClassifier(class_weight='balanced',
               colsample_bytree=np.float64(0.8569542256977045),
               learning_rate=np.float64(0.0529666151588077), max_depth=8,
               min_child_samples=120, n_estimators=661, n_jobs=-1,
               num_leaves=17, objective='binary', random_state=42,
               reg_alpha=np.float64(1.5215700972337949),
               reg_lambda=np.float64(1.1225543951389925),
               subsample=np.float64(0.9198385129840964), verbosity=-1)

In [47]:
# Attention : le seuil doit avoir été déterminé avant le réentraînement final.
holdout_probabilities = (
    final_model.predict_proba(
        X_holdout
    )[:, 1]
)

holdout_predictions = (
    holdout_probabilities
    >= optimal_threshold
).astype("int8")

In [48]:
holdout_metrics, _, _ = evaluate_classifier(
    final_model,
    X_holdout,
    y_holdout,
    threshold=optimal_threshold
)

holdout_metrics

{'threshold': 0.5,
 'roc_auc': 0.7922703496168028,
 'accuracy': 0.7486504650204869,
 'precision': 0.1965607649599013,
 'recall': 0.6844790547798066,
 'f1': 0.30541576803259046,
 'business_cost': 22169,
 'normalized_business_cost': 0.48060788692089235,
 'tn': 31984,
 'fp': 10419,
 'fn': 1175,
 'tp': 2549}

### Run 7 — Enregistrer le modèle final dans MLflow

In [49]:
with mlflow.start_run(
    run_name="07_final_lightgbm_model"
) as final_run:

    mlflow.set_tags({
        "project": "credit_scoring",
        "stage": "candidate_model",
        "model_family": "gradient_boosting",
        "dataset_version": "train_features_v1",
        "description": (
            "Modèle final sélectionné après optimisation "
            "des hyperparamètres et du seuil métier."
        )
    })

    mlflow.log_params(
        final_model.get_params()
    )

    mlflow.log_param(
        "decision_threshold",
        optimal_threshold
    )

    mlflow.log_param(
        "false_negative_cost",
        10
    )

    mlflow.log_param(
        "false_positive_cost",
        1
    )

    mlflow.log_metrics({
        f"holdout_{key}": value
        for key, value
        in holdout_metrics.items()
    })

    # Sauvegarde du seuil comme artefact
    threshold_config = {
        "decision_threshold": optimal_threshold,
        "false_negative_cost": 10,
        "false_positive_cost": 1,
        "positive_class": 1
    }

    threshold_path = (
        ARTIFACTS_DIR
        / "decision_threshold.json"
    )

    with open(
        threshold_path,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            threshold_config,
            file,
            indent=2
        )

    mlflow.log_artifact(
        str(threshold_path),
        artifact_path="configuration"
    )

    model_info = mlflow.sklearn.log_model(
        sk_model=final_model,
        name="model",
        input_example=X_holdout.head(5)
    )

    print("Run final :", final_run.info.run_id)
    print("Model URI :", model_info.model_uri)

Run final : 7da8c96b487745518bc7a93baedd715a
Model URI : models:/m-ab572f4437ff4444876cfdab42fd72b2
🏃 View run 07_final_lightgbm_model at: http://127.0.0.1:5000/#/experiments/1/runs/7da8c96b487745518bc7a93baedd715a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


La fonction mlflow.sklearn.log_model permet d’enregistrer un modèle Scikit-learn dans MLflow ; un modèle peut ensuite être enregistré dans le Model Registry directement ou à partir de son URI.

## Afficher les résultats des runs depuis Python

In [50]:
experiment = mlflow.get_experiment_by_name(
    experiment_name
)

runs = mlflow.search_runs(
    experiment_ids=[
        experiment.experiment_id
    ]
)

columns_to_display = [
    "run_id",
    "tags.mlflow.runName",
    "metrics.default_roc_auc",
    "metrics.optimal_business_cost",
    "metrics.validation_roc_auc",
    "metrics.holdout_roc_auc",
    "params.model"
]

existing_columns = [
    column
    for column in columns_to_display
    if column in runs.columns
]

display(
    runs[existing_columns]
    .sort_values(
        by=[
            column
            for column in [
                "metrics.default_roc_auc"
            ]
            if column in existing_columns
        ],
        ascending=False
    )
)

,run_id,tags.mlflow.runName,metrics.default_roc_auc,metrics.optimal_business_cost,metrics.validation_roc_auc,metrics.holdout_roc_auc,params.model
4,928dd53fc9024826b949993cabc163ee,04_lightgbm_baseline,0.790133,29585.0,NaN,NaN,None
6,be44797797b84be2ae9367a5eebfcc42,02_logistic_regression_baseline,0.775608,30944.0,NaN,NaN,LogisticRegression
8,d5a52d076bec4ac783a73cb3dc64b1c5,02_logistic_regression_baseline,0.775608,30944.0,NaN,NaN,LogisticRegression
0,7da8c96b487745518bc7a93baedd715a,07_final_lightgbm_model,NaN,NaN,NaN,0.79227,None
1,efcf6c1fa27a46d29cbd6658c1d7fab8,06_lightgbm_randomized_search,NaN,NaN,0.790893,NaN,None
2,660b52f2da11487cb635f951cb612f33,05_lightgbm_class_weight_balanced,NaN,29425.0,NaN,NaN,None
3,8aad1493d4874d75a48f3b6fb5a659ab,05_lightgbm_class_weight_None,NaN,29357.0,NaN,NaN,None
5,6f743c19e4564e24acaa2935e14ab1b7,03_logistic_regression_cross_validation,NaN,NaN,NaN,NaN,LogisticRegression
7,c67a736aa0074b899cba540d54c9210a,01_dummy_baseline,NaN,NaN,NaN,NaN,DummyClassifier
9,5ba9186ad7254b41b0af527bef4bef28,01_dummy_baseline,NaN,NaN,NaN,NaN,DummyClassifier


## Run 8 - Utiliser mlflow.autolog()

Une fois le logging manuel maîtrisé, tu peux tester :

In [51]:
mlflow.sklearn.autolog(
    log_input_examples=True,
    log_model_signatures=True,
    log_models=True,
    silent=False
)

In [52]:
with mlflow.start_run(
    run_name="08_logistic_autolog_test"
):

    test_model = LogisticRegression(
        max_iter=500,
        class_weight="balanced",
        random_state=RANDOM_STATE
    )

    test_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            test_model
        )
    ])

    test_pipeline.fit(
        X_train,
        y_train
    )

    probabilities = test_pipeline.predict_proba(
        X_valid
    )[:, 1]

    mlflow.log_metric(
        "custom_validation_auc",
        roc_auc_score(
            y_valid,
            probabilities
        )
    )

🏃 View run 08_logistic_autolog_test at: http://127.0.0.1:5000/#/experiments/1/runs/7211405bc56049afbff89b7696eda396
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


MLflow propose à la fois le logging manuel et l’autologging ; le logging manuel donne davantage de contrôle sur les métriques métier, tags et artefacts personnalisés.

Après ce test, désactive l’autologging pour éviter des logs involontaires :

In [53]:
mlflow.sklearn.autolog(
    disable=True
)